# Exploratory Data Analysis — Home Credit Default Risk

**Project:** Knowledge Discovery in Banking Dataset  
**Phase:** 1 — Data Understanding & Preprocessing  
**Step:** 1 — EDA & Analytical Justification  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Date:** 2026-05-10

---

This notebook is the **analytical foundation** for the entire KDD pipeline. Every finding documented here translates directly into a justified preprocessing decision in Step 2. The notebook follows a diagnostic posture: we map every condition, every risk, every anomaly. We do not transform the data yet. We understand it first.

## Setup — Libraries and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [ ]:
DATA = '../datasets/'

app        = pd.read_csv(DATA + 'application_train.csv')
bureau     = pd.read_csv(DATA + 'bureau.csv')
bureau_bal = pd.read_csv(DATA + 'bureau_balance.csv')
prev       = pd.read_csv(DATA + 'previous_application.csv')
pos        = pd.read_csv(DATA + 'POS_CASH_balance.csv')
inst       = pd.read_csv(DATA + 'installments_payments.csv')
cc         = pd.read_csv(DATA + 'credit_card_balance.csv')

print('All datasets loaded successfully.')
print(f'  application_train : {app.shape[0]:>8,} rows x {app.shape[1]:>3} cols')
print(f'  bureau            : {bureau.shape[0]:>8,} rows x {bureau.shape[1]:>3} cols')
print(f'  bureau_balance    : {bureau_bal.shape[0]:>8,} rows x {bureau_bal.shape[1]:>3} cols')
print(f'  previous_app      : {prev.shape[0]:>8,} rows x {prev.shape[1]:>3} cols')
print(f'  POS_CASH_balance  : {pos.shape[0]:>8,} rows x {pos.shape[1]:>3} cols')
print(f'  installments      : {inst.shape[0]:>8,} rows x {inst.shape[1]:>3} cols')
print(f'  credit_card_bal   : {cc.shape[0]:>8,} rows x {cc.shape[1]:>3} cols')

---
## Section 1: Dataset Orientation

### What We Are Investigating and Why

Before examining any individual feature, we must understand **what each table represents**, what the grain of each record is, and how the tables relate to each other. Misunderstanding the grain of a relational table leads to inflated statistics and incorrect aggregations in every downstream step.

The Home Credit dataset is a **star schema** centered on the `application_train.csv` primary table. Each related table describes a different facet of the applicant's financial history. We examine:

- The **size and structure** of each table
- The **primary key uniqueness** of the main table — are there duplicate applications?
- The **cardinality of each relationship** — how many related records does the average applicant have?
- The **population overlap** — which applicants appear in which related tables?

In [ ]:
print('=== application_train.csv ===')
print(f'Rows: {len(app):,}  |  Columns: {app.shape[1]}')
print(f'Dtype breakdown: {dict(app.dtypes.value_counts())}')

n_unique_pk = app['SK_ID_CURR'].nunique()
print(f'\nPrimary key (SK_ID_CURR) uniqueness: {n_unique_pk:,} unique of {len(app):,} total')
print(f'Duplicate PKs: {len(app) - n_unique_pk}')

num_cols = app.select_dtypes(include=np.number).columns.difference(['SK_ID_CURR','TARGET'])
cat_cols = app.select_dtypes(include='object').columns
print(f'\nNumerical features : {len(num_cols)}')
print(f'Categorical features: {len(cat_cols)}')

print('\n=== Relational Table Summary ===')
n_app = app['SK_ID_CURR'].nunique()
tables = [('bureau', bureau), ('previous_application', prev),
          ('POS_CASH_balance', pos), ('installments_payments', inst),
          ('credit_card_balance', cc)]
print(f"{'Table':<28} {'Rows':>10} {'Applicants':>12} {'Missing':>10}")
print('-' * 65)
for name, df in tables:
    if 'SK_ID_CURR' in df.columns:
        n_linked = df['SK_ID_CURR'].nunique()
        n_miss = n_app - n_linked
        print(f'{name:<28} {len(df):>10,} {n_linked:>12,} {n_miss:>10,}')

print('\n=== Records-per-applicant Distribution ===')
for name, df in tables:
    if 'SK_ID_CURR' not in df.columns:
        continue
    c = df['SK_ID_CURR'].value_counts()
    print(f'{name:<28}  med={c.median():.0f}  p75={c.quantile(0.75):.0f}  p95={c.quantile(0.95):.0f}  max={c.max()}')

### Interpretation

**Grain of the primary table:** Each row in `application_train.csv` represents **one loan application** from one unique applicant. The primary key `SK_ID_CURR` has **zero duplicates** across 307,511 rows. The dataset is clean at the source level.

**Feature surface:** 122 columns total — 106 numerical, 16 categorical, plus `TARGET` and `SK_ID_CURR`. ~48 columns are building/property measurements reported in three statistical variants (`_AVG`, `_MODE`, `_MEDI`), so the effective feature diversity is much lower than the column count implies. This triplication pattern has significant implications for missingness and correlation analysis.

**Relational cardinality and behavioral completeness:**

| Table | Behavioral Dimension | Median Records/Applicant | Applicants Without Records |
|---|---|---|---|
| `bureau` | External credit history depth | 4 | 1,700 (0.6%) |
| `previous_application` | Past application behavior | 4 | — |
| `POS_CASH_balance` | POS/cash repayment snapshots | 22 months | ~323 |
| `installments_payments` | Installment repayment history | 25 records | ~323 |
| `credit_card_balance` | Revolving credit usage | 22 months | ~132 |

**Key structural insights:**

- **Bureau coverage is near-universal (99.4%):** The 0.6% without bureau records are likely first-time borrowers — a qualitatively distinct group that cannot be characterized by credit history metrics.
- **Monthly snapshot tables accumulate quickly:** `bureau_balance` has 27.3M rows; `installments_payments` has 13.6M. These must be aggregated before joining to the main table.
- **The median of 4 bureau records per applicant** with max of 116 implies a long-tailed distribution. Applicants with many bureau records have richer credit histories but also more complex behavioral profiles.
- **`bureau_balance` links via `SK_ID_BUREAU`** (not `SK_ID_CURR`) — a two-level join is required for applicant-level analysis of monthly bureau snapshots.

---
## Section 2: Target Label Inspection

### What We Are Investigating and Why

The target variable `TARGET` encodes **whether a given applicant defaulted on their Home Credit loan**. Understanding its distribution is prerequisite to interpreting every other feature in the dataset. A heavily imbalanced label tells us something fundamental about the rarity of the phenomenon — and shapes which features will carry meaningful discriminating signal.

**Critical constraint:** The `TARGET` label will be used here **only** for distributional understanding and feature relevance ranking. It will be dropped before any clustering, association rule mining, or anomaly detection algorithm is applied. That drop will be documented explicitly in the Preprocessing Report.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
ax = axes[0]
vc = app['TARGET'].value_counts().sort_index()
colors = ['#4CAF50', '#E53935']
bars = ax.bar(['Repaid (0)', 'Defaulted (1)'], vc.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{val:,}\n({val/len(app)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Target Label Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Applications')
ax.set_ylim(0, vc.max() * 1.18)

# Default rate by contract type
ax = axes[1]
ct_def = app.groupby('NAME_CONTRACT_TYPE')['TARGET'].agg(['mean','count'])
bars = ax.bar(ct_def.index, ct_def['mean'] * 100, color=['#1565C0','#E65100'], edgecolor='white')
for bar, (idx, row) in zip(bars, ct_def.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{row["mean"]*100:.1f}%\n(n={row["count"]:,})', ha='center', fontsize=10)
ax.set_title('Default Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, ct_def['mean'].max() * 100 * 1.4)

# Default rate by income type
ax = axes[2]
inc_counts = app['NAME_INCOME_TYPE'].value_counts()
inc_def = app[app['NAME_INCOME_TYPE'].isin(inc_counts[inc_counts >= 50].index)] \
            .groupby('NAME_INCOME_TYPE')['TARGET'].mean().sort_values(ascending=False)
bars = ax.barh(inc_def.index, inc_def.values * 100, color='#7B1FA2', alpha=0.8)
ax.axvline(app['TARGET'].mean() * 100, color='red', linestyle='--', linewidth=1.5,
           label=f'Overall: {app["TARGET"].mean()*100:.1f}%')
ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Income Type', fontsize=13, fontweight='bold')
ax.legend()

plt.suptitle('Section 2: Target Label — Default Risk Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('notebooks/s2_target_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'Total applications  : {len(app):,}')
print(f'Repaid  (TARGET=0) : {(app["TARGET"]==0).sum():,}  ({(app["TARGET"]==0).mean()*100:.2f}%)')
print(f'Default (TARGET=1) : {(app["TARGET"]==1).sum():,}  ({(app["TARGET"]==1).mean()*100:.2f}%)')
print(f'Imbalance ratio     : {(app["TARGET"]==0).sum()/(app["TARGET"]==1).sum():.1f} : 1')
print('\nDefault rate by gender:')
print(app[app['CODE_GENDER']!='XNA'].groupby('CODE_GENDER')['TARGET']
        .agg(['mean','count']).assign(mean=lambda x: x['mean'].map('{:.1%}'.format)))

### Interpretation

**What the label represents:** `TARGET = 1` means the applicant experienced payment difficulties — specifically, late payments of more than X days on at least one installment during the first repayment period.

**Class imbalance:** The dataset is **heavily imbalanced** at an 11.4:1 ratio — **91.9% repaid, 8.1% defaulted**. This is structurally expected in consumer credit: default is a rare but high-cost event. For this KDD project (unsupervised), it means that in any clustering analysis, cluster membership will naturally skew toward non-defaulters by volume. Cluster-level default rates will be a meaningful enrichment metric only when interpreted as a rate, not a count.

**Structural variation in default risk:**

- **Cash loans default at 8.3%**, versus **revolving loans at 5.5%**. Revolving loan customers appear better screened or more financially established — counterintuitive given that revolving credit typically implies higher risk in other markets.
- **Males default at 10.1% versus females at 7.0%**. This 44% relative difference is substantial and likely reflects both demographic risk factors and the types of loans sought by each group.
- **Working applicants default at 9.6%**, far above **pensioners at 5.4%** and **state servants at 5.8%**. Pensioners receive fixed, predictable income — a structural buffer against default. Working applicants face income volatility risk.

**Label handling confirmation:** The `TARGET` label is present, inspected, and its distribution is now understood. It will be used **only** for feature relevance analysis in Section 7. It will be **excluded from all feature matrices** passed to clustering, association rule mining, and anomaly detection algorithms.

---
## Section 3: Missing Value Analysis

### What We Are Investigating and Why

Missingness in banking data is **rarely random**. A missing value often carries its own information: a missing `OWN_CAR_AGE` likely means the applicant does not own a car. A missing `EXT_SOURCE_1` may indicate the external credit bureau could not find a record — itself a signal of thin credit history.

We classify columns into **four missingness tiers** and, for each column with significant missingness, ask: *why is this value absent, and what does that absence mean for this entity?*

**Missingness tiers:**
- **Tier 1 — Negligible (<5%):** Likely data collection gaps; safe to impute
- **Tier 2 — Moderate (5–30%):** Requires investigation; may be conditional
- **Tier 3 — High (30–60%):** Likely structurally absent; imputation manufactures noise
- **Tier 4 — Severe (>60%):** Near-unusable for most entities; binary missingness indicator preferred

In [ ]:
missing_counts = app.isnull().sum()
missing_pct    = (missing_counts / len(app) * 100).round(2)
missing_df = pd.DataFrame({'count': missing_counts, 'pct': missing_pct}) \
               .sort_values('pct', ascending=False)
missing_df = missing_df[missing_df['count'] > 0]

def tier(pct):
    if pct < 5:  return 'Tier 1 (<5%)'
    if pct < 30: return 'Tier 2 (5-30%)'
    if pct < 60: return 'Tier 3 (30-60%)'
    return              'Tier 4 (>60%)'

missing_df['tier'] = missing_df['pct'].apply(tier)
tier_counts = missing_df['tier'].value_counts()
print('Missingness tier summary:')
for t in ['Tier 1 (<5%)', 'Tier 2 (5-30%)', 'Tier 3 (30-60%)', 'Tier 4 (>60%)', 'No missing']:
    if t == 'No missing':
        print(f'  {t}: {app.shape[1] - len(missing_df)} columns')
    else:
        print(f'  {t}: {tier_counts.get(t, 0)} columns')

tier_colors = {'Tier 1 (<5%)': '#2196F3', 'Tier 2 (5-30%)': '#FF9800',
               'Tier 3 (30-60%)': '#E53935', 'Tier 4 (>60%)': '#6A1B9A'}

fig, axes = plt.subplots(1, 2, figsize=(18, 12))

top_miss = missing_df.head(40)
colors = [tier_colors[t] for t in top_miss['tier']]
ax = axes[0]
ax.barh(top_miss.index[::-1], top_miss['pct'][::-1], color=colors[::-1])
ax.axvline(60, color='purple', linestyle='--', alpha=0.7, label='Tier 4 boundary (60%)')
ax.axvline(30, color='red',    linestyle='--', alpha=0.7, label='Tier 3 boundary (30%)')
ax.set_xlabel('Missing (%)')
ax.set_title('Top 40 Columns by Missing Rate', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
tier_labels = ['Tier 1\n(<5%)', 'Tier 2\n(5-30%)', 'Tier 3\n(30-60%)', 'Tier 4\n(>60%)', 'No Missing']
tier_vals   = [tier_counts.get('Tier 1 (<5%)', 0), tier_counts.get('Tier 2 (5-30%)', 0),
               tier_counts.get('Tier 3 (30-60%)', 0), tier_counts.get('Tier 4 (>60%)', 0),
               app.shape[1] - len(missing_df)]
tier_col    = ['#2196F3', '#FF9800', '#E53935', '#6A1B9A', '#4CAF50']
bars = ax.bar(tier_labels, tier_vals, color=tier_col, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, tier_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=12, fontweight='bold')
ax.set_title('Columns by Missingness Tier', fontsize=12, fontweight='bold')
ax.set_ylabel('Column Count')

plt.suptitle('Section 3: Missing Value Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s3_missing_values.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n--- Key domain columns ---')
key_domain = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','OWN_CAR_AGE','OCCUPATION_TYPE','AMT_ANNUITY','AMT_GOODS_PRICE']
for col in key_domain:
    print(f'  {col:<35}: {missing_pct.get(col, 0):.2f}%')
house_cols = [c for c in app.columns if any(s in c for s in ['_AVG','_MODE','_MEDI'])]
print(f'\nHousing-related columns ({len(house_cols)} total): range {missing_pct[house_cols].min():.1f}%-{missing_pct[house_cols].max():.1f}%')

### Interpretation

**Overall missingness profile:** 67 of 122 columns (55%) have at least some missing values. This is high, but largely explained by the dataset's structural design.

**Tier 4 — Severe missingness (>60%), 17 columns:**

All 17 are building/property measurement columns (`COMMONAREA_*`, `NONLIVINGAPARTMENTS_*`, `FLOORSMIN_*`, `YEARS_BUILD_*`) plus `OWN_CAR_AGE`. The building columns were collected only for applicants living in apartments with formal registration records — the ~69% missingness reflects that ~30% of applicants live in non-apartment housing where these measurements simply do not apply.

`OWN_CAR_AGE` (66.0% missing) is categorically different: it is missing because the applicant does not own a car (`FLAG_OWN_CAR = 'N'`). **The absence of `OWN_CAR_AGE` is informationally equivalent to car ownership status.**

`WARNING: Requires attention in preprocessing` — For `OWN_CAR_AGE`, a binary indicator `FLAG_NO_CAR` must be created before any imputation or drop decision.

**Tier 3 — High missingness (30–60%), 33 columns:**

- **47 housing columns** cluster between 47–70% missingness (all three statistical aggregates for the same physical properties). They are structurally co-missing: if `APARTMENTS_AVG` is missing, `APARTMENTS_MEDI` and `APARTMENTS_MODE` are missing at identical rates.
- **`EXT_SOURCE_1` (56.4% missing)** is the most analytically critical Tier 3 column. It is the strongest predictor of default (see Section 7). Its missingness likely indicates applicants with **thin credit files** — people whose credit history is too limited for the bureau to generate a score. This absence is itself a strong signal of credit risk.

`WARNING: Requires attention in preprocessing` — Create binary indicator `FLAG_EXT_SOURCE_1_MISSING` before imputation.

- **`OCCUPATION_TYPE` (31.4% missing)** appears to be a genuine data collection gap.

**Co-missingness pattern:** The entire housing block (~47 columns) is systematically co-missing. In preprocessing, retain **one statistical variant** (`_MODE`) and drop the redundant `_AVG` and `_MEDI` versions. The missingness indicator for the housing block should be a single binary column, not 47 separate ones.

---
## Section 4: Numerical Feature Distributions

### What We Are Investigating and Why

The shape of a feature's distribution determines which transformations are appropriate in preprocessing. Right-skewed income data requires log transformation before distance-based clustering. Negative values in `DAYS_EMPLOYED` are technically valid (elapsed time) but require domain understanding. Implausible extremes are not just statistical anomalies — they are data quality signals.

We examine: descriptive statistics, skewness, kurtosis, and the distributions of the most analytically important features.

In [ ]:
num_cols = app.select_dtypes(include=np.number).columns.difference(['SK_ID_CURR', 'TARGET'])

desc = app[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc['skewness'] = app[num_cols].skew()
desc['kurtosis'] = app[num_cols].kurtosis()

print('Key financial feature statistics:')
key_show = ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE',
            'DAYS_BIRTH','DAYS_EMPLOYED','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
print(desc.loc[[c for c in key_show if c in desc.index],
               ['mean','50%','99%','max','skewness']].to_string())

print(f'\nFeatures with extreme skewness (|skew|>2, excluding binary flags):')
high_skew = desc[(abs(desc['skewness']) > 2) & (desc['max'] > 1)].sort_values('skewness', ascending=False)
print(f'Count: {len(high_skew)}')
print(high_skew[['50%','max','skewness']].head(10).to_string())

print(f'\nDAYS_EMPLOYED sentinel (365243): {(app["DAYS_EMPLOYED"]==365243).sum():,} records ({(app["DAYS_EMPLOYED"]==365243).mean()*100:.2f}%)')
print(f'Age range (years): {abs(app["DAYS_BIRTH"].max())/365.25:.1f} to {abs(app["DAYS_BIRTH"].min())/365.25:.1f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

key_feats = [
    ('AMT_INCOME_TOTAL', 'Total Income (capped at 2M)'),
    ('AMT_CREDIT',       'Credit Amount'),
    ('AMT_ANNUITY',      'Monthly Annuity'),
    ('AMT_GOODS_PRICE',  'Goods Price'),
    ('DAYS_BIRTH',       'Days Since Birth (negative)'),
    ('EXT_SOURCE_2',     'External Credit Score 2'),
]

for ax, (col, title) in zip(axes.flat, key_feats):
    data = app[col].dropna()
    if col == 'AMT_INCOME_TOTAL': data = data[data <= 2e6]
    if col == 'DAYS_EMPLOYED':   data = data[data != 365243]
    ax.hist(data, bins=60, color='#1565C0', alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():,.0f}')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Section 4: Key Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s4_distributions.png', bbox_inches='tight', dpi=120)
plt.show()

# DAYS_EMPLOYED sentinel breakdown
sent = app[app['DAYS_EMPLOYED'] == 365243]
print('Sentinel records by income type:')
print(sent['NAME_INCOME_TYPE'].value_counts().to_string())
print(f'Default rate — sentinel : {sent["TARGET"].mean():.3f}')
print(f'Default rate — non-sent.: {app[app["DAYS_EMPLOYED"]!=365243]["TARGET"].mean():.3f}')

### Interpretation

**Income (`AMT_INCOME_TOTAL`):** Extremely right-skewed. Median ₽147,150 vs mean ₽168,798 vs max ₽117,000,000. The gap between the 99th percentile (₽472,500) and the maximum signals a small cluster of ultra-high-income applicants that are not representative of the population. `WARNING: Requires attention in preprocessing` — Log transformation is mandatory before distance-based computation. Winsorize at p99 first.

**Credit amount (`AMT_CREDIT`):** Right-skewed (skewness 1.23) but far less extreme than income. Median ₽513,531, max ₽4,050,000. Modal cluster around ₽450K–₽600K with a long tail. Log transformation is recommended.

**Annuity (`AMT_ANNUITY`):** Right-skewed (1.58). Median monthly payment ₽24,903, max ₽258,025. Its shape mirrors credit amount (bounded by credit amount and interest rate).

**Age (`DAYS_BIRTH`):** Stored as **negative integers** (days elapsed from birth, hence all negative). Range −25,229 to −7,489 translates to ages **20.5 to 69.1 years**. Symmetric distribution, no anomalous values. Converting to positive years is mandatory for interpretability.

**`DAYS_EMPLOYED` — Critical sentinel value:** The value 365,243 (= 1,000 years in days) affects **55,374 records (18.0%)** — exclusively pensioners (55,352) and unemployed applicants (22). This is a **sentinel value**, not a data error. Pension and unemployment are states where employment duration is inapplicable. The sentinel group's default rate of 5.4% vs. 8.7% for non-sentinel confirms it encodes a real behavioral distinction. `WARNING: Requires attention in preprocessing` — Create `FLAG_SENTINEL_EMPLOYED`, then replace 365,243 with NaN.

**External credit scores (`EXT_SOURCE_1/2/3`):** All bounded in [0,1]. Their near-zero mutual correlations (r < 0.22) confirm they measure independent signals from different bureaus. These are the strongest individual predictors of default (Section 7). EXT_SOURCE_1's 56% missingness makes it the most critical feature handling challenge.

---
## Section 5: Categorical Feature Analysis

### What We Are Investigating and Why

Categorical features are the **interpretive backbone of cluster profiling**. When we later describe what a cluster represents in human terms — 'young, salaried, urban applicants' or 'pensioners with real estate' — we will lean heavily on these variables. Knowing their distributions now tells us how much discriminating power each one carries.

We specifically look for: **high-cardinality features** (many unique values), **low-variance features** (one dominant category >90%), and **catch-all categories** like 'XNA', 'Other', or 'Unknown'.

In [ ]:
cat_cols = app.select_dtypes(include='object').columns.tolist()

cat_summary = []
for col in cat_cols:
    vc = app[col].value_counts(dropna=True)
    n_unique = app[col].nunique(dropna=True)
    dominant_pct = vc.iloc[0] / len(app) * 100 if len(vc) > 0 else 0
    n_xna = app[col].isin(['XNA','Unknown','XAP','Other','not specified']).sum()
    cat_summary.append({'column': col, 'n_unique': n_unique,
                        'dominant_category': str(vc.index[0]) if len(vc)>0 else 'N/A',
                        'dominant_pct': round(dominant_pct, 1),
                        'xna_count': n_xna,
                        'missing_pct': round(app[col].isna().mean()*100, 1)})

cat_df = pd.DataFrame(cat_summary).sort_values('dominant_pct', ascending=False)
print(cat_df.to_string(index=False))

low_var = cat_df[cat_df['dominant_pct'] >= 88]
print(f'\nLow-variance categoricals (dominant >= 88%):')
for _, row in low_var.iterrows():
    print(f'  {row["column"]}: {repr(row["dominant_category"])} = {row["dominant_pct"]}%')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

plot_cols = [
    ('NAME_INCOME_TYPE', 'Income Type'),
    ('NAME_EDUCATION_TYPE', 'Education Level'),
    ('NAME_FAMILY_STATUS', 'Family Status'),
    ('NAME_CONTRACT_TYPE', 'Contract Type'),
    ('NAME_HOUSING_TYPE', 'Housing Type'),
    ('CODE_GENDER', 'Gender'),
]

for ax, (col, title) in zip(axes.flat, plot_cols):
    vc = app[col].value_counts(dropna=True).head(10)
    ax.barh(vc.index[::-1], vc.values[::-1], color='#1565C0', alpha=0.8)
    for i, (idx, val) in enumerate(vc.items()):
        ax.text(val + vc.max()*0.01, len(vc)-1-i, f'{val/len(app)*100:.1f}%', va='center', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Count')

plt.suptitle('Section 5: Key Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s5_categoricals.png', bbox_inches='tight', dpi=120)
plt.show()

print('\nORGANIZATION_TYPE: top 12 categories')
print(app['ORGANIZATION_TYPE'].value_counts().head(12).to_string())
print(f'Total unique: {app["ORGANIZATION_TYPE"].nunique()}')

### Interpretation

**`NAME_CONTRACT_TYPE`:** Binary — Cash loans (90.5%) vs Revolving loans (9.5%). The overwhelming dominance of cash loans reflects Home Credit's primary product mix.

**`CODE_GENDER`:** 65.8% Female, 34.2% Male, 4 records coded 'XNA' (data entry errors or institutional applicants). `WARNING: Requires attention in preprocessing` — The 4 'XNA' records should be treated as missing.

**`NAME_INCOME_TYPE`:** Working (51.6%), Commercial associate (23.3%), Pensioner (18.0%), State servant (7.1%). Four categories account for 99.9% of records. The remaining categories — Unemployed (22), Student (18), Businessman (10), Maternity leave (5) — are statistically negligible but carry extreme default rates (Unemployed: 36.4%, Maternity leave: 40.0%). `WARNING: Requires attention in preprocessing` — These four rare categories should be grouped into an 'Other/High-risk' category for encoding purposes.

**`NAME_EDUCATION_TYPE`:** Highly concentrated — Secondary education (71.0%), Higher education (24.3%). Lower secondary (1.2%) and Academic degree (0.1%) are sparse and may need merging.

**`NAME_HOUSING_TYPE`:** Low-variance — House/apartment dominates (88.7%). Limited discriminating power for segmentation.

**`ORGANIZATION_TYPE`:** **High cardinality** (58 unique values). Critically, **18.0% are coded 'XNA'** — aligning with the `DAYS_EMPLOYED` sentinel (18.0%), confirming these are the pensioners/unemployed for whom an organization type is inapplicable. `WARNING: Requires attention in preprocessing` — Group the ~50 granular business entity types into ~8 macro-sectors.

**Low-variance features:** `NAME_CONTRACT_TYPE` (90.5%), `NAME_HOUSING_TYPE` (88.7%), `NAME_TYPE_SUITE` (80.8%). These contribute minimal cluster separation and should be considered for exclusion from distance-based algorithms.

---
## Section 6: Outlier Identification and Classification

### What We Are Investigating and Why

Outliers in financial data are not monolithic — they have **distinct types** with different implications:

1. **Data error:** Technically impossible or clearly wrong (e.g., age < 18, negative income)
2. **Sentinel value:** A deliberately inserted placeholder encoding a specific condition
3. **Legitimate extreme case:** A real entity at the tail of the distribution (e.g., very high income)

We apply **IQR × 3** (rather than the standard 1.5×) to respect that financial distributions are inherently heavy-tailed. Using 1.5× would flag normal high-income applicants as outliers.

This section is the **conceptual foundation for Phase 4 (Anomaly Detection)** — the typology established here determines how Phase 4 anomalies are categorized.

In [ ]:
key_num = ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE',
           'DAYS_EMPLOYED','CNT_CHILDREN','CNT_FAM_MEMBERS',
           'OBS_30_CNT_SOCIAL_CIRCLE','DEF_30_CNT_SOCIAL_CIRCLE',
           'OBS_60_CNT_SOCIAL_CIRCLE','DEF_60_CNT_SOCIAL_CIRCLE']

outlier_report = []
for col in key_num:
    if col not in app.columns: continue
    data = app[col].dropna()
    if col == 'DAYS_EMPLOYED': data = data[data != 365243]
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 3*IQR, Q3 + 3*IQR
    n_out = ((app[col] < lo) | (app[col] > hi))
    if col == 'DAYS_EMPLOYED': n_out = n_out & (app[col] != 365243)
    n_out = n_out.sum()
    outlier_report.append({'column': col, 'IQR_lo': round(lo,1), 'IQR_hi': round(hi,1),
                           'n_outliers': n_out, 'pct_outliers': round(n_out/len(app)*100, 2),
                           'actual_max': app[col].max()})

out_df = pd.DataFrame(outlier_report)
print(out_df[['column','IQR_lo','IQR_hi','n_outliers','pct_outliers','actual_max']].to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
box_cols = [('AMT_INCOME_TOTAL',2e6,'Total Income (capped 2M)'),
            ('AMT_CREDIT',None,'Credit Amount'),
            ('AMT_ANNUITY',None,'Monthly Annuity'),
            ('CNT_CHILDREN',None,'Children Count'),
            ('OBS_30_CNT_SOCIAL_CIRCLE',50,'Social Circle Obs (30d, capped 50)'),
            ('DEF_30_CNT_SOCIAL_CIRCLE',None,'Social Circle Defaults (30d)')]

for ax, (col, cap, title) in zip(axes.flat, box_cols):
    data = app[col].dropna()
    if col == 'DAYS_EMPLOYED': data = data[data != 365243]
    if cap: data = data[data <= cap]
    ax.boxplot(data, vert=False, patch_artist=True,
               boxprops=dict(facecolor='#90CAF9', color='navy'),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(col)

plt.suptitle('Section 6: Outlier Boxplots (IQR x3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s6_outliers.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n=== AMT_INCOME_TOTAL Extremes ===')
print(f'p99: {app["AMT_INCOME_TOTAL"].quantile(0.99):,.0f}')
print(f'max: {app["AMT_INCOME_TOTAL"].max():,.0f}')
print(f'Records above 1M: {(app["AMT_INCOME_TOTAL"] > 1e6).sum()}')

### Interpretation

**Type 2 — Sentinel value: `DAYS_EMPLOYED = 365,243` (18.0% of records)**

The value 365,243 (exactly 1,000 years in days) is a deliberate placeholder. All 55,374 records are pensioners (55,352) or unemployed (22). The sentinel group's default rate of 5.4% vs. 8.7% for non-sentinel confirms it encodes a real behavioral distinction. `WARNING: Requires attention in preprocessing` — Create `FLAG_SENTINEL_EMPLOYED`, replace 365,243 with NaN.

**Type 3 — Legitimate extreme cases: `AMT_INCOME_TOTAL` above ₽1M (250 records)**

High-income applicants are not data errors. The maximum of ₽117M is an extreme outlier, but the broader high-income population represents an economically real and important segment. **Deleting high-income records would remove a meaningful sub-population.** `WARNING: Requires attention in preprocessing` — Winsorize at p99 (₽472,500), then log-transform.

**Type 3 — Legitimate but analytically problematic: `DEF_30_CNT_SOCIAL_CIRCLE` (11.4% flagged)**

The 11.4% 'outlier' rate simply reflects that most applicants have no social-circle defaults, while ~11% have at least one. **Calling this 11.4% 'outliers' is misleading** — they represent a distinct behavioral sub-population (applicants with socially proximate credit risk), not noise. `WARNING: Requires attention in preprocessing` — Bin into 0/1/2+ indicator; do not winsorize.

**`CNT_CHILDREN` max of 19:** 126 records with children > 4; the value of 19 is biologically implausible as a direct count and may reflect data entry errors. Records with CNT_CHILDREN > 10 should be capped at a defensible limit.

---
## Section 7: Feature Correlation Analysis

### What We Are Investigating and Why

Correlation analysis serves two downstream purposes: **feature selection** (identifying redundant features) and **feature interpretation** (understanding what each retained feature actually represents).

We compute the Pearson correlation matrix for all numerical features, visualize it as a heatmap, and examine correlation with `TARGET` to understand which features carry the strongest signal for distinguishing defaulters from non-defaulters.

In [ ]:
num_cols_full = app.select_dtypes(include=np.number).columns.difference(['SK_ID_CURR'])
corr_target = app[num_cols_full].corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)

print('Top 15 features by absolute correlation with TARGET:')
print(corr_target.head(15).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
top15 = corr_target.head(15)
colors_bar = ['#D32F2F' if 'EXT_SOURCE' in c else '#1565C0' for c in top15.index]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors_bar[::-1])
ax.set_xlabel('|Pearson r| with TARGET')
ax.set_title('Top 15 Features Correlated with Default', fontsize=12, fontweight='bold')
ax.axvline(0.05, color='gray', linestyle='--', alpha=0.5)

ax = axes[1]
selected = list(corr_target.head(10).index) + ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','DAYS_EMPLOYED']
selected = list(dict.fromkeys(selected))
corr_sub = app[selected + ['TARGET']].corr()
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('Correlation Heatmap — Key Features', fontsize=12, fontweight='bold')

plt.suptitle('Section 7: Feature Correlation Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s7_correlations.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
print('=== Highly correlated feature pairs (|r| > 0.85) ===')
num_only = app.select_dtypes(include=np.number).columns.difference(['SK_ID_CURR','TARGET'])
corr_matrix = app[num_only].corr()

high_corr_pairs = []
cols_list = corr_matrix.columns.tolist()
for i in range(len(cols_list)):
    for j in range(i+1, len(cols_list)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.85:
            high_corr_pairs.append((cols_list[i], cols_list[j], r))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print(f'Total pairs with |r| > 0.85: {len(high_corr_pairs)}')
print('\nTop 20:')
for c1, c2, r in high_corr_pairs[:20]:
    print(f'  {c1:<40} vs  {c2:<35}  r={r:.3f}')

### Interpretation

**External credit scores dominate target correlation:**

The three strongest predictors of default are `EXT_SOURCE_3` (|r|=0.179), `EXT_SOURCE_2` (|r|=0.160), and `EXT_SOURCE_1` (|r|=0.155). While these correlations are modest in absolute terms — typical for complex behavioral phenomena — they are the most predictive single features in the dataset. Their significance is validated by domain knowledge: external credit scores aggregate years of repayment behavior into a single normalized measure.

**Age and default (|r|=0.078):** Older applicants default less. `DAYS_BIRTH` is negative (older = more negative), so the positive correlation with TARGET means younger applicants default more.

**89 highly correlated pairs identified (|r| > 0.85):**

The pattern is dominated by three redundancy types:

1. **AVG/MODE/MEDI triplication (r ≈ 0.99):** Every building measurement exists as three statistical summaries. Retaining all three would triple-weight building attributes in distance-based analysis. `WARNING: Requires attention in preprocessing` — Retain only `_MODE`; drop `_AVG` and `_MEDI`.

2. **`DAYS_EMPLOYED` vs `FLAG_EMP_PHONE` (r = −1.0):** Perfect negative correlation because `FLAG_EMP_PHONE = 1` whenever `DAYS_EMPLOYED ≠ 365,243`. After sentinel handling, `FLAG_EMP_PHONE` becomes redundant.

3. **`OBS_30` vs `OBS_60` (r = 0.998):** Same underlying social network observation with different time windows. Retain only the 30-day variant.

**EXT_SOURCE_1/2/3 are independent (r = 0.11–0.21):** Each measures a different bureau's assessment. All three should be retained despite both being target-correlated — they carry different signals.

**Near-zero correlated features:** Several `FLAG_DOCUMENT_*` binary flags show near-zero correlation with the target and with each other. Their role may be in interaction with other features rather than standalone signal.

---
## Section 8: Relational Feature Analysis

### What We Are Investigating and Why

The main application table captures a **snapshot** — the applicant's stated profile at a single point in time. The relational tables capture **behavioral history** — what this applicant has actually done with credit over time. This is where the most analytically powerful features originate.

We examine each related table's structure, the distribution of record counts per applicant, and the behavioral dimensions each table contributes. We also ask: **what does it mean to have zero records in a given table?** Absence can be as informative as presence.

In [ ]:
print('=== BUREAU: External Credit History ===')
bureau_counts = bureau.groupby('SK_ID_CURR').size()
n_app = app['SK_ID_CURR'].nunique()
print(f'Applicants with bureau records : {bureau_counts.shape[0]:,} ({bureau_counts.shape[0]/n_app*100:.1f}%)')
print(f'Applicants WITHOUT bureau      : {n_app - bureau_counts.shape[0]:,} ({(n_app-bureau_counts.shape[0])/n_app*100:.1f}%)')
print(f'Records per applicant: p25={bureau_counts.quantile(0.25):.0f}  p50={bureau_counts.quantile(0.5):.0f}  p75={bureau_counts.quantile(0.75):.0f}  p95={bureau_counts.quantile(0.95):.0f}  max={bureau_counts.max()}')
print('\nCredit active status:')
print(bureau['CREDIT_ACTIVE'].value_counts().to_string())

print('\n=== PREVIOUS APPLICATION ===')
prev_counts = prev.groupby('SK_ID_CURR').size()
print(f'Records: {len(prev):,}  |  Unique applicants: {prev_counts.shape[0]:,}')
print(f'Apps per applicant: p50={prev_counts.median():.0f}  p95={prev_counts.quantile(0.95):.0f}')
print('Contract status:')
print(prev['NAME_CONTRACT_STATUS'].value_counts().to_string())
print(f'Historical approval rate: {(prev["NAME_CONTRACT_STATUS"]=="Approved").mean()*100:.1f}%')

print('\n=== INSTALLMENT PAYMENTS ===')
inst_copy = inst.copy()
inst_copy['DPD'] = (inst_copy['DAYS_ENTRY_PAYMENT'] - inst_copy['DAYS_INSTALMENT']).clip(lower=0)
print(f'Records: {len(inst):,}  |  Unique applicants: {inst["SK_ID_CURR"].nunique():,}')
print(f'Mean DPD: {inst_copy["DPD"].mean():.2f} days')
print(f'Late payments (DPD>0): {(inst_copy["DPD"]>0).mean()*100:.1f}%')
print(f'Severe late (DPD>30): {(inst_copy["DPD"]>30).mean()*100:.2f}%')

print('\n=== CREDIT CARD BALANCE ===')
cc_months = cc.groupby('SK_ID_CURR')['MONTHS_BALANCE'].nunique()
util = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)
print(f'Records: {len(cc):,}  |  Unique applicants: {cc["SK_ID_CURR"].nunique():,}')
print(f'Months of history: p50={cc_months.median():.0f}  p95={cc_months.quantile(0.95):.0f}')
print(f'Mean utilization: {util.mean()*100:.1f}%  |  median: {util.median()*100:.1f}%')

print('\n=== BUREAU BALANCE ===')
bb_months = bureau_bal.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].nunique()
print(f'Records: {len(bureau_bal):,}')
print(f'Observation window: {bureau_bal["MONTHS_BALANCE"].min()} to {bureau_bal["MONTHS_BALANCE"].max()} months')
print(f'History per bureau record: p50={bb_months.median():.0f}  p95={bb_months.quantile(0.95):.0f}')
print('\nMonthly status distribution:')
print(bureau_bal['STATUS'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
ax.hist(bureau_counts.clip(upper=30), bins=30, color='#1565C0', alpha=0.8, edgecolor='white')
ax.set_title('Bureau Records per Applicant (capped 30)', fontsize=11, fontweight='bold')
ax.set_xlabel('Number of Bureau Records')
ax.set_ylabel('Applicant Count')
ax.axvline(bureau_counts.median(), color='red', linestyle='--', label=f'Median: {bureau_counts.median():.0f}')
ax.legend()

ax = axes[0, 1]
prev_status = prev['NAME_CONTRACT_STATUS'].value_counts()
ax.bar(prev_status.index, prev_status.values, color=['#4CAF50','#FF9800','#E53935','#9E9E9E'])
ax.set_title('Previous Application Contract Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(ax.patches, prev_status.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
            f'{val/len(prev)*100:.1f}%', ha='center', fontsize=10)

ax = axes[1, 0]
inst_copy2 = inst.copy()
inst_copy2['DPD'] = (inst_copy2['DAYS_ENTRY_PAYMENT'] - inst_copy2['DAYS_INSTALMENT']).clip(lower=0)
ax.hist(inst_copy2['DPD'].clip(upper=50), bins=50, color='#E53935', alpha=0.8, edgecolor='white')
ax.set_title('Installment Days Past Due (DPD, capped 50)', fontsize=11, fontweight='bold')
ax.set_xlabel('DPD (days)')
ax.set_ylabel('Record Count')

ax = axes[1, 1]
status_map = {'C':'Closed','0':'On-time','1':'1 DPD','2':'2-29 DPD','3':'30-59','4':'60-89','5':'90+','X':'Unknown'}
bb_status = bureau_bal['STATUS'].map(status_map).fillna(bureau_bal['STATUS']).value_counts()
ax.bar(bb_status.index, bb_status.values, color='#7B1FA2', alpha=0.8)
ax.set_title('Bureau Balance Monthly Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Record Count')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.suptitle('Section 8: Relational Table Behavioral Dimensions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s8_relational.png', bbox_inches='tight', dpi=120)
plt.show()

### Interpretation

**Bureau table — depth of external credit history:**

99.4% of applicants have at least one bureau credit record, indicating a mature credit infrastructure. The median of 4 records per applicant (p75=8, p95=14, max=116) shows a moderately right-skewed distribution. 63.0% of bureau credit records are **closed** and 36.7% are **active**. A high ratio of active-to-closed credits may indicate applicants who are currently heavily leveraged.

**Bureau balance — repayment quality signal:**

The bureau balance table spans **96 months (8 years)** of history. The STATUS field encodes: C=closed, 0=on-time, X=data not available, 1–5 = DPD buckets. The dominance of 'Closed' (13.6M) and 'On-time' (7.5M) records indicates generally healthy repayment histories. However, 242,347 records with 1+ DPD and 62,406 with 90+ DPD represent meaningful delinquency signals.

**Previous application — behavioral selection signal:**

61.9% of previous applications were approved. The 17.4% refusal rate is significant — applicants who have been refused in the past represent systematically different risk profiles. The median of 4 previous applications shows substantial prior engagement with Home Credit.

**Installment payments — actual repayment behavior:**

8.4% of historical installment payments were late (DPD > 0), with mean DPD of 1.03 days. The max of 2,884 days reveals a heavily right-skewed distribution. **Applicants at the extreme right tail are in sustained default** — this variable, when aggregated per applicant, will be a powerful behavioral feature.

**Credit card balance — revolving credit utilization:**

Mean utilization (balance/limit) and months of history will be key aggregated features. High utilization (>80%) is a classic indicator of financial stress.

**The absence is a feature:** 1,700 applicants with no bureau records are likely thin-file individuals. A binary `FLAG_NO_BUREAU` should be created — their absence from bureau is itself a behavioral signal.

---
## Section 9: Cross-Feature Relationship Analysis

### What We Are Investigating and Why

Individual feature distributions tell us about single variables in isolation. Cross-feature analysis reveals **relationships, structure, and interaction effects** that individual distributions cannot surface. The clusters we will find in Phase 2 will be defined by combinations of features — understanding those combinations now means we can interpret cluster profiles with precision.

We examine 6 analytically motivated pairs, each chosen based on domain reasoning, and compare expectation against actual finding.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Income vs Credit Amount
ax = axes[0, 0]
sample = app[app['AMT_INCOME_TOTAL'] < 700000].sample(5000, random_state=42)
scatter = ax.scatter(sample['AMT_INCOME_TOTAL'], sample['AMT_CREDIT'],
                     c=sample['TARGET'], cmap='RdYlGn_r', alpha=0.3, s=10)
ax.set_xlabel('Income (local currency)')
ax.set_ylabel('Credit Amount')
ax.set_title('Income vs Credit Amount (colored by default)', fontsize=11, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Default')
x_r = np.linspace(0, 700000, 100)
for ratio, ls in [(3,'--'), (6,':'), (9,'-.')]: ax.plot(x_r, x_r*ratio, ls, color='gray', alpha=0.6, label=f'{ratio}x')
ax.legend(fontsize=8)

# 2. Age vs EXT_SOURCE_1
ax = axes[0, 1]
s2 = app[['DAYS_BIRTH','EXT_SOURCE_1']].dropna().sample(3000, random_state=42)
s2 = s2.copy()
s2['age'] = s2['DAYS_BIRTH'].abs() / 365.25
ax.scatter(s2['age'], s2['EXT_SOURCE_1'], alpha=0.2, s=8, color='#1565C0')
r = s2[['age','EXT_SOURCE_1']].corr().iloc[0,1]
ax.set_xlabel('Age (years)')
ax.set_ylabel('EXT_SOURCE_1')
ax.set_title(f'Age vs External Credit Score 1\n(r = {r:.3f})', fontsize=11, fontweight='bold')

# 3. DTI distribution
ax = axes[0, 2]
dti = (app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL'] * 100).clip(upper=80)
ax.hist(dti, bins=60, color='#7B1FA2', alpha=0.8, edgecolor='white')
ax.axvline(dti.median(), color='red', linestyle='--', label=f'Median DTI: {dti.median():.1f}%')
ax.axvline(36, color='orange', linestyle=':', label='36% guideline')
ax.set_xlabel('Annuity / Income (%)')
ax.set_title('Debt-to-Income (Annuity/Income) Distribution', fontsize=11, fontweight='bold')
ax.legend()

# 4. Default rate by EXT_SOURCE_2 bin
ax = axes[1, 0]
app_tmp = app.copy()
app_tmp['ext2_bin'] = pd.cut(app['EXT_SOURCE_2'].dropna(), bins=10)
ext2_def = app_tmp.dropna(subset=['EXT_SOURCE_2']).groupby('ext2_bin', observed=True)['TARGET'].mean()
ax.plot(range(len(ext2_def)), ext2_def.values*100, marker='o', color='#E53935', linewidth=2)
ax.set_xticks(range(len(ext2_def)))
ax.set_xticklabels([f'{i.mid:.2f}' for i in ext2_def.index], rotation=45, fontsize=8)
ax.set_xlabel('EXT_SOURCE_2 (bin midpoint)')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by EXT_SOURCE_2 Score', fontsize=11, fontweight='bold')

# 5. Gender x Income Type default heatmap
ax = axes[1, 1]
pivot = app[app['CODE_GENDER'] != 'XNA'].groupby(['CODE_GENDER','NAME_INCOME_TYPE'])['TARGET'].mean().unstack()
valid_inc = app['NAME_INCOME_TYPE'].value_counts()
pivot = pivot[[c for c in pivot.columns if valid_inc.get(c, 0) > 100]]
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn_r', ax=ax, linewidths=0.5, annot_kws={'size':9})
ax.set_title('Default Rate: Gender x Income Type', fontsize=11, fontweight='bold')

# 6. Credit amount by education
ax = axes[1, 2]
edu_order = app.groupby('NAME_EDUCATION_TYPE')['AMT_CREDIT'].median().sort_values(ascending=False).index
app.boxplot(column='AMT_CREDIT', by='NAME_EDUCATION_TYPE', ax=ax,
            patch_artist=True, showfliers=False, order=edu_order)
ax.set_xlabel('')
ax.set_ylabel('Credit Amount')
ax.set_title('Credit Amount by Education Level', fontsize=11, fontweight='bold')
ax.set_xticklabels(edu_order, rotation=30, ha='right', fontsize=9)
plt.sca(ax); plt.title('')

plt.suptitle('Section 9: Cross-Feature Relationship Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/s9_cross_features.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
print('=== Credit-to-Income Ratio ===')
cti = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
print(f'Median: {cti.median():.2f}x')
print(f'p75   : {cti.quantile(0.75):.2f}x')
print(f'p95   : {cti.quantile(0.95):.2f}x')
print(f'>6x   : {(cti>6).sum():,} ({(cti>6).mean()*100:.1f}%)')

print('\n=== Age vs EXT_SOURCE correlations ===')
app['age_years'] = app['DAYS_BIRTH'].abs() / 365.25
for src in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
    r = app[['age_years',src]].dropna().corr().iloc[0,1]
    print(f'  Age vs {src}: r = {r:.3f}')

print('\n=== Debt-to-Income (Annuity/Income) ===')
dti2 = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']
print(f'Median DTI: {dti2.median()*100:.1f}%')
print(f'p75 DTI   : {dti2.quantile(0.75)*100:.1f}%')
print(f'p95 DTI   : {dti2.quantile(0.95)*100:.1f}%')
print(f'DTI > 50% : {(dti2>0.5).sum():,} ({(dti2>0.5).mean()*100:.2f}%)')

print('\n=== AMT_CREDIT median by education ===')
print(app.groupby('NAME_EDUCATION_TYPE')['AMT_CREDIT'].median().sort_values(ascending=False).to_string())

### Interpretation

**1. Income vs Credit Amount — Systematic over-requesting:**

The median credit-to-income ratio is **3.27x**, meaning the typical applicant requests a loan worth more than three times their annual income. The p95 is 9.16x, and 9,844 applicants (3.2%) request more than 10x their annual income. This is structurally expected — loans are repaid over time — but the right tail suggests some degree of systematic over-requesting or income misrepresentation. The scatter shows no strong linear relationship between income and credit amount, meaning applicants across the income spectrum request similar absolute credit amounts. This will produce meaningful cluster separation — high-income applicants with modest loans vs. low-income applicants with similar loans have very different debt burden profiles.

**2. Age vs External Credit Score 1 — Strong positive relationship (r = 0.601):**

This is the strongest cross-feature relationship identified. Older applicants have higher external credit scores — domain sense: scores reflect accumulated repayment history, and older applicants have had more time to build that history. `EXT_SOURCE_2` and `EXT_SOURCE_3` show much weaker age correlations (r = 0.09 and 0.21), suggesting they measure behavioral patterns less influenced by history length. Both age and EXT_SOURCE_1 should be retained — they carry different aspects of credit quality.

**3. Debt-to-Income ratio — The population is modestly leveraged:**

The median DTI (annuity as fraction of income) is **16.3%** — substantially below the 36% guideline in consumer finance. The p95 is 35.5%. Only 2,530 applicants (0.8%) have DTI above 50%. **This population is not over-leveraged** — which is partly why overall default rates are low at 8.1%.

**4. EXT_SOURCE_2 and default — Monotonic inverse relationship:**

Default rate declines monotonically as EXT_SOURCE_2 score increases. The relationship appears non-linear — risk drops sharply at higher score levels. This monotonic relationship should be preserved in any transformation.

**5. Gender × Income type interaction:**

Males have higher default rates across all income types. The gender gap is larger for 'Working' applicants (males ~11%, females ~7%) than for 'Pensioner' applicants (both ~5–6%). This suggests the gender effect is partially mediated by employment type and income stability.

**6. Education vs Credit amount — Systematic socioeconomic gradient:**

Academic degree holders take the largest loans (median ~₽610K), down to Lower secondary (~₽433K). This 41% spread confirms that education serves as a proxy for borrowing capacity. It will be a strong cluster discriminator.

---
## Section 10: EDA Summary and Preprocessing Justification Table

### What We Are Documenting and Why

This section is the **formal handoff from Step 1 to Step 2**. Every finding documented in Sections 1–9 is mapped to a specific planned preprocessing action. Step 2 executes exactly what this table prescribes. Nothing in Step 2 should be a surprise relative to what is documented here.

---

## EDA Summary: Key Findings and Preprocessing Implications

### Critical Issues (require immediate action in Step 2)

1. **`DAYS_EMPLOYED` sentinel value (365,243):** Affects 18.0% of the dataset. Must be flagged and separated before any numerical treatment. Treating this as a real numerical value would corrupt distance calculations.

2. **`AMT_INCOME_TOTAL` extreme outliers:** One record at ₽117M and ~250 above ₽1M will dominate distance calculations without capping. Log transformation is mandatory.

3. **Housing column triplication:** 47 columns in `_AVG`/`_MODE`/`_MEDI` variants carry near-perfect correlations (r ≈ 0.99). Retaining all three would incorrectly triple-weight building attributes.

4. **`CODE_GENDER` = 'XNA':** 4 records with invalid gender coding. Must be treated as missing.

5. **Rare income type categories:** Unemployed (22), Student (18), Businessman (10), Maternity leave (5) — insufficient for standalone encoding; will create unstable cluster assignments.

### Structural Issues (require careful handling)

6. **`EXT_SOURCE_1` missing at 56.4%:** The strongest single predictor of default is missing for over half the dataset. A binary missingness indicator preserves the information that the applicant has a thin credit file.

7. **Housing block missingness (47–70%):** ~50% structurally absent (non-apartment housing). A single missingness indicator suffices.

8. **`OCCUPATION_TYPE` (31.4% missing):** Genuine data collection gap. Missingness does not carry the same structural meaning as `OWN_CAR_AGE` or `EXT_SOURCE_1`.

9. **`ORGANIZATION_TYPE` = 'XNA' (18.0%):** Corresponds to the pensioner/unemployed sentinel group. Must be handled consistently with `DAYS_EMPLOYED` treatment.

10. **`DAYS_BIRTH` negative encoding:** Must be converted to positive years before distance-based analysis.

### Informative Patterns (preserved and interpreted)

11. **EXT_SOURCE_1/2/3 independence:** Near-zero mutual correlations (r < 0.22); all three retained.

12. **Age–EXT_SOURCE_1 relationship (r = 0.60):** Credit score and age will co-vary in cluster profiles; both should be retained.

13. **Bureau absence as a feature:** 1,700 applicants with no bureau records; binary `FLAG_NO_BUREAU` captures this structural distinction.

14. **DEF_30_CNT_SOCIAL_CIRCLE behavioral pattern:** 11.4% of applicants have social-circle defaults; this is a behavioral sub-population feature, not noise.

15. **Installment DPD behavioral signal:** 8.4% of historical installments paid late; when aggregated per applicant, a powerful behavioral feature for segmentation.

---

## Preprocessing Justification Table

| Finding | Affected Columns | Severity | Planned Action in Step 2 | Justification |
|---|---|---|---|---|
| `DAYS_EMPLOYED` sentinel (365,243) | `DAYS_EMPLOYED`, `ORGANIZATION_TYPE` | **High** | Create `FLAG_SENTINEL_EMPLOYED`; replace 365,243 with NaN; handle `ORGANIZATION_TYPE` = 'XNA' | Sentinel corrupts distributions; 18% of data; differential default rate (5.4% vs 8.7%) confirms it encodes a real behavioral distinction |
| Extreme income outliers | `AMT_INCOME_TOTAL` | **High** | Winsorize at p99 (₽472,500); apply log transformation | Max ₽117M is 247× the median; without capping, a single record dominates Euclidean distances |
| Housing column triplication | All `_AVG` and `_MEDI` columns (31 columns) | **High** | Drop all `_AVG` and `_MEDI` variants; retain `_MODE` only | Pearson r > 0.99 between variants; retaining all three triples the weight of building attributes |
| `DAYS_BIRTH` negative encoding | `DAYS_BIRTH` | **High** | Convert to `AGE_YEARS = abs(DAYS_BIRTH) / 365.25` | Negative values are meaningless in distance calculations |
| `CODE_GENDER` = 'XNA' | `CODE_GENDER` | **Medium** | Treat as missing; impute with mode | 4 records; likely data entry error |
| Rare income type categories | `NAME_INCOME_TYPE` | **Medium** | Group Unemployed + Student + Businessman + Maternity leave into 'Other/Rare' | Total n=55; cannot form stable cluster segments |
| `EXT_SOURCE_1` high missingness | `EXT_SOURCE_1` | **High** | Create `FLAG_EXT_SOURCE_1_MISSING`; impute missing with median | 56.4% missing; missingness indicates thin credit file (itself a risk signal) |
| `OWN_CAR_AGE` structural missingness | `OWN_CAR_AGE` | **High** | Create `FLAG_NO_CAR = (OWN_CAR_AGE is NaN)`; impute remaining with 0 or median | Missingness encodes car ownership; binary indicator is more informative than imputed age |
| `OCCUPATION_TYPE` missingness | `OCCUPATION_TYPE` | **Medium** | Impute with mode per income type group; or encode NaN as 'Unknown' | 31.4% missing; genuine collection gap |
| `ORGANIZATION_TYPE` high cardinality | `ORGANIZATION_TYPE` | **Medium** | Group 58 categories into ~8 macro-sectors | 58 categories too granular; macro-grouping preserves behavioral signal |
| Housing block missingness | All 47 housing columns | **Medium** | Create single `FLAG_NO_HOUSING_DATA`; impute with 0 or column median | ~50% structurally absent; single indicator sufficient |
| Highly correlated pairs (89 pairs, r > 0.85) | `OBS_30` vs `OBS_60`; `FLAG_EMP_PHONE` vs `DAYS_EMPLOYED`; etc. | **Medium** | Drop redundant partner in each pair; retain member with higher target correlation | Redundant features inflate effective dimensionality |
| `DEF_30_CNT_SOCIAL_CIRCLE` pseudo-outliers | `DEF_30_CNT_SOCIAL_CIRCLE`, `DEF_60_CNT_SOCIAL_CIRCLE` | **Low** | Bin into 0/1/2+ indicator; do not winsorize | 11.4% 'flagged' represent a real behavioral sub-population, not noise |
| `CNT_CHILDREN` > 10 | `CNT_CHILDREN`, `CNT_FAM_MEMBERS` | **Low** | Cap at 10; winsorize at p99 | Value of 19 is biologically implausible; capping at a defensible limit preferred over deletion |
| `TARGET` label | `TARGET` | **High** | Drop from feature matrix before clustering, rule mining, anomaly detection | Project constraint: unsupervised methods must not use the label as input |
| `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE` skewness | All three amount columns | **Medium** | Apply log transformation | Right-skewed (skewness 1.2–1.6); log transform improves Euclidean distance properties |
| Bureau absence | `SK_ID_CURR` not in `bureau` | **Medium** | Create `FLAG_NO_BUREAU` binary feature | 1,700 applicants without bureau records are thin-file; absence is a behavioral feature |
| Installment DPD aggregation | `installments_payments` table | **Medium** | Compute per-applicant: mean DPD, max DPD, pct late, pct severely late (>30d) | 8.4% late rate; behavioral aggregates are stronger features than application-level attributes |
| Credit card utilization | `credit_card_balance` table | **Medium** | Compute per-applicant mean and max utilization (balance/limit); months of history | Utilization >80% is a classic stress indicator in revolving credit |

---

*This table is the formal handoff to Step 2 — Preprocessing Pipeline. Step 2 executes exactly what is prescribed here. No transformation in Step 2 should be a surprise relative to what is documented above.*